# 01 — Random Forest Baseline

Train a Random Forest classifier on hand-crafted spectral features (CCF metrics,
line-depth ratios, broadening indicators, etc.) as a baseline for comparison with
the 1D CNN that operates on raw spectra.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.inspection import permutation_importance
import joblib

# path setup — ensure project root is importable
os.chdir(os.path.join(os.path.dirname(os.path.abspath(".")), ".."))
sys.path.insert(0, os.getcwd())

from src.models import build_rf_pipeline
from src.utils import plot_roc_pr, plot_confusion_matrix, print_metrics, save_figure
from config import MODEL_CONFIG, VIS_CONFIG

%matplotlib inline
plt.rcParams["figure.dpi"] = VIS_CONFIG["dpi"]

## 1. Load Features and Labels

In [ ]:
# load hand-crafted features
df = pd.read_parquet("data/handcrafted_features.parquet")
print(f"Feature table shape: {df.shape}")
print(f"\nFeature columns:\n{list(df.columns)}")

# separate features and labels
label_col = "is_binary"
feature_cols = [c for c in df.columns if c not in (label_col, "APOGEE_ID")]
X = df[feature_cols].copy()
y = df[label_col].values

print(f"\nClass balance:")
print(f"  Single (0): {(y == 0).sum()}  ({(y == 0).mean():.1%})")
print(f"  Binary (1): {(y == 1).sum()}  ({(y == 1).mean():.1%})")

# handle NaN values with median imputation
n_nan = X.isna().sum().sum()
print(f"\nTotal NaN values: {n_nan}")
if n_nan > 0:
    from sklearn.impute import SimpleImputer
    imputer = SimpleImputer(strategy="median")
    X = pd.DataFrame(imputer.fit_transform(X), columns=feature_cols)
    print("Imputed NaN values with column medians.")

## 2. Train/Test Split

In [ ]:
rs = MODEL_CONFIG["random_state"]
test_size = MODEL_CONFIG["test_size"]
val_size = MODEL_CONFIG["val_size"]

# first split: train+val vs test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=test_size, stratify=y, random_state=rs
)

# second split: train vs val
relative_val = val_size / (1 - test_size)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=relative_val,
    stratify=y_trainval, random_state=rs
)

print(f"Train: {len(y_train)}  (binary: {y_train.sum()})")
print(f"Val:   {len(y_val)}  (binary: {y_val.sum()})")
print(f"Test:  {len(y_test)}  (binary: {y_test.sum()})")
print(f"Split: {len(y_train)/len(y):.0%} / {len(y_val)/len(y):.0%} / {len(y_test)/len(y):.0%}")

## 3. Train Random Forest

In [ ]:
# build and train the RF pipeline
pipeline = build_rf_pipeline()
pipeline.fit(X_train, y_train)

print(f"RF trained: {MODEL_CONFIG['rf_n_estimators']} trees, "
      f"max_depth={MODEL_CONFIG['rf_max_depth']}, "
      f"class_weight={MODEL_CONFIG['rf_class_weight']}")

# 5-fold stratified cross-validation on training set
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=rs)
cv_scores = cross_val_score(
    pipeline, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1
)
print(f"\n5-fold CV AUROC: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
print(f"  Per-fold: {[f'{s:.4f}' for s in cv_scores]}")

## 4. Evaluate on Test Set

In [ ]:
# predict on test set
y_pred = pipeline.predict(X_test)
y_probs = pipeline.predict_proba(X_test)[:, 1]

# print full classification report and AUROC
auroc = print_metrics(y_test, y_pred, y_probs)

# confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
plot_confusion_matrix(y_test, y_pred, ax=ax)
save_figure(fig, "rf_confusion_matrix")
plt.show()

In [ ]:
# ROC and precision-recall curves
fig = plot_roc_pr(y_test, {"Random Forest": y_probs})
save_figure(fig, "rf_roc_pr")
plt.show()

## 5. Feature Importance

In [ ]:
# permutation importance on the test set
perm_imp = permutation_importance(
    pipeline, X_test, y_test,
    n_repeats=10, random_state=rs, scoring="roc_auc", n_jobs=-1
)

# sort and plot top 20 features
sorted_idx = perm_imp.importances_mean.argsort()[::-1][:20]
top_features = np.array(feature_cols)[sorted_idx]
top_means = perm_imp.importances_mean[sorted_idx]
top_stds = perm_imp.importances_std[sorted_idx]

fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(range(len(top_features)), top_means[::-1], xerr=top_stds[::-1],
        align="center", color="steelblue", edgecolor="k", linewidth=0.5)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features[::-1])
ax.set_xlabel("Permutation Importance (decrease in AUROC)")
ax.set_title("Top 20 Features — Permutation Importance")
plt.tight_layout()
save_figure(fig, "rf_feature_importance")
plt.show()

## 6. Save Model

In [ ]:
# save the trained pipeline
os.makedirs("data", exist_ok=True)
joblib.dump(pipeline, "data/rf_model.pkl")
print("Saved RF model to data/rf_model.pkl")

# save test predictions for the comparison notebook
np.savez(
    "data/rf_test_preds.npz",
    y_true=y_test,
    y_pred=y_pred,
    y_probs=y_probs,
)
print("Saved test predictions to data/rf_test_preds.npz")